<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 25 — רגרסיה משולבת וניתוח גודל הטעות
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
מחיר מחשב תלוי בכמה מדידות ולעיתים בקטגוריה. נחבר הכנה משולבת לרגרסיה ונמדוד טעות ביחידות השימוש.

## מטרות
- לשלב סוגי מאפיינים.
- לחשב MAE.
- להשוות מדדי רגרסיה לבסיס.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## המושגים הנחוצים
אחרי הכנה, רגרסיה משתמשת בייצוג התכונות המעובד. MAE הוא ממוצע טעויות מוחלטות וביחידות של y. ברגרסיה לא מבצעים stratify לפי ערכי יעד רציף בודדים. משווים לקו בסיס באותו פרוטוקול. תקנון אינו הכרחי לרגרסיה ליניארית רגילה ללא רגולריזציה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## דוגמה פתורה לפני קוד
אמת [100,200,300] ותחזית [110,180,330] נותנות טעויות מוחלטות 10,20,30. ה־MAE הוא 20. ממוצע שאריות חתומות אינו המדד הזה.

</div>

In [ ]:
import numpy as np
actual = np.array([100, 200, 300])
predicted = np.array([110, 180, 330])
mae = np.mean(np.abs(actual - predicted))
print(mae)

<div dir="rtl" lang="he" style="text-align: right;">

**פלט צפוי:** 20.0

## מדוע זה חשוב?
סכום משוקלל, שגיאה ועדכון משקולות מכינים ל־DL. רגרסיה ופרספטרון אינם מוצגים כאלגוריתמי חובה ביחידה 5.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## יישום מודרך — חלק 1
מקור הקוד: ערכה 39 הקיימת. בהמשך עוברים לתרגול עצמאי באותו נושא.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

### תשתית הנתונים והחלוקה
קראו תחילה את היעד ואת שמות המאפיינים. החלוקה שומרת התאמה בין שורות X ל־y. הפרמטר random_state מאפשר שחזור ואינו משפר דיוק בפני עצמו.

</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score,mean_absolute_error
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
raw=pd.read_csv(url)
data=raw.dropna(subset=["body_mass_g"]).copy()
numeric=["bill_length_mm","flipper_length_mm"]
categorical=["island"]
X=data[numeric+categorical].copy()
y=data["body_mass_g"].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42)
print("Eligible:",len(data),"Train:",len(X_train),"Test:",len(X_test))

<div dir="rtl" lang="he" style="text-align: right;">

### אימות בתוך הפיתוח
ההכנה נמצאת בתוך Pipeline. כל קיפול לומד את פרמטריו מתת־האימון שלו. נקרא את ממוצע האימות לפני שנחליט על מודל.

</div>

In [ ]:
prepare=ColumnTransformer([
 ("numeric",Pipeline([("fill",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric),
 ("categorical",Pipeline([
  ("fill",SimpleImputer(strategy="most_frequent")),
  ("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
 ]),categorical)
])
model=Pipeline([("prepare",prepare),("regression",LinearRegression())])
cv=KFold(n_splits=5,shuffle=True,random_state=42)
cv_scores=cross_val_score(model,X_train,y_train,cv=cv,scoring="r2")
print("CV R² mean:",cv_scores.mean(),"Descriptive std:",cv_scores.std(ddof=0))

<div dir="rtl" lang="he" style="text-align: right;">

### התאמה ויישום
זהו מה לומד פרמטרים ובאילו שורות נעשה fit. ודאו שכל הכנה שנלמדת מנתונים משתמשת באימון המתאים. לאחר מכן פירשו את הפלט במילים.

</div>

In [ ]:
model.fit(X_train,y_train)
predictions=model.predict(X_test)
comparison=pd.DataFrame({"actual_g":y_test,"predicted_g":predictions},index=y_test.index)
comparison["residual_g"]=comparison["actual_g"]-comparison["predicted_g"]
mae_manual=comparison["residual_g"].abs().mean()
mae=mean_absolute_error(y_test,predictions)
r2=r2_score(y_test,predictions)
baseline=DummyRegressor(strategy="mean")
baseline.fit(X_train,y_train)
baseline_predictions=baseline.predict(X_test)
baseline_mae=mean_absolute_error(y_test,baseline_predictions)
baseline_r2=r2_score(y_test,baseline_predictions)
display(comparison.head(8))
print("Test R²:",r2,"MAE (g):",mae,"Manual MAE:",mae_manual)
print("Training-mean baseline R²:",baseline_r2,"MAE (g):",baseline_mae)

<div dir="rtl" lang="he" style="text-align: right;">

### המחשה חזותית
לפני הרצה הגדירו שאלה, סוגי משתנים ויחידות. אחרי הרצה פרשו את הצירים ואת הממצא. אין להסיק סיבתיות או למחוק קצה רק מן הציור.

</div>

In [ ]:
plt.figure()
plt.scatter(y_test,predictions,alpha=.65)
limits=[min(y_test.min(),predictions.min()),max(y_test.max(),predictions.max())]
plt.plot(limits,limits,color="black",label="Perfect agreement line")
plt.xlabel("Actual mass (g)");plt.ylabel("Predicted mass (g)")
plt.title("Test predictions — multiple features");plt.legend()
plt.tight_layout();plt.show()

<div dir="rtl" lang="he" style="text-align: right;">

## עצירה לפני המעבדה
האם MAE=20 פירושו 20 אחוזים?

**התוצר הנדרש:** מאפיינים אחרים, הכנה משולבת, KFold, MAE ו־R² מול קו בסיס. זוהי משימת העברה של תהליך הרגרסיה.

</div>